In [ ]:
!pip install statsbombpy pandas numpy xgboost scipy shap scikit-learn --quiet

import numpy as np
import pandas as pd
from statsbombpy import sb
from xgboost import XGBClassifier
from sklearn.model_selection import GroupKFold
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss
import shap
import warnings
warnings.filterwarnings('ignore')

# Pitch Goal Coordinates (StatsBomb Scale)
P_LEFT = np.array([120.0, 36.0])
P_RIGHT = np.array([120.0, 44.0])

def calc_bisector_distance(shot_pos, gk_pos):
    """Calculates perpendicular distance from GK to shot-to-posts angle bisector."""
    S, GK = np.array(shot_pos, dtype=float), np.array(gk_pos, dtype=float)
    v_l, v_r = P_LEFT - S, P_RIGHT - S
    norm_l, norm_r = np.linalg.norm(v_l), np.linalg.norm(v_r)
    if norm_l == 0 or norm_r == 0:
        return 0.0
    u, v = v_l / norm_l, v_r / norm_r
    b = u + v
    norm_b = np.linalg.norm(b)
    if norm_b == 0:
        return 0.0
    b_hat = b / norm_b
    w = GK - S
    return abs(w[0] * b_hat[1] - w[1] * b_hat[0])

def count_occlusion_defenders(shot_pos, gk_pos, defender_locs, buffer_radius=1.5):
    """Counts defenders within a line-of-sight corridor between shooter and GK."""
    S, GK = np.array(shot_pos, dtype=float), np.array(gk_pos, dtype=float)
    v = GK - S
    v_norm = np.linalg.norm(v)
    if v_norm == 0:
        return 0
    u = v / v_norm
    count = 0
    for d_loc in defender_locs:
        D = np.array(d_loc, dtype=float)
        w = D - S
        proj = np.dot(w, u)
        if 0 <= proj <= v_norm:
            if np.linalg.norm(w - proj * u) <= buffer_radius:
                count += 1
    return count

print("1. Pulling StatsBomb Open Data...")
comps = sb.competitions()
shots_data = []

for _, comp in comps.iterrows():
    try:
        matches = sb.matches(competition_id=comp['competition_id'], season_id=comp['season_id'])
        # Retain match_id for GroupKFold splitting
        for match_id in matches['match_id'].unique():
            events = sb.events(match_id=match_id)
            shots = events[events['type'] == 'Shot'].copy()

            for _, shot in shots.iterrows():
                freeze_frame = shot.get('shot_freeze_frame')
                shot_loc = shot.get('location')

                if not isinstance(freeze_frame, list) or not shot_loc or len(shot_loc) < 2:
                    continue

                gk_loc, gk_name = None, None
                defender_locs = []

                for actor in freeze_frame:
                    is_teammate = actor.get('teammate', False)
                    pos = actor.get('position', {})
                    pos_name = pos.get('name') if isinstance(pos, dict) else str(pos)

                    if pos_name == 'Goalkeeper' and not is_teammate:
                        gk_loc = actor.get('location')
                        p_info = actor.get('player', {})
                        gk_name = p_info.get('name') if isinstance(p_info, dict) else str(p_info)
                    elif not is_teammate:
                        loc = actor.get('location')
                        if loc:
                            defender_locs.append(loc)

                if gk_loc is None or gk_name is None or gk_name == 'None':
                    continue

                v_l, v_r = P_LEFT - np.array(shot_loc), P_RIGHT - np.array(shot_loc)
                cos_theta = np.clip(np.dot(v_l, v_r) / (np.linalg.norm(v_l) * np.linalg.norm(v_r)), -1.0, 1.0)
                shot_angle = np.degrees(np.arccos(cos_theta))

                shot_dist = np.linalg.norm(np.array([120.0, 40.0]) - np.array(shot_loc))
                bisector_dist = calc_bisector_distance(shot_loc, gk_loc)
                gk_depth = 120.0 - gk_loc[0]
                occlusion_count = count_occlusion_defenders(shot_loc, gk_loc, defender_locs, buffer_radius=1.5)

                p_xg = shot.get('shot_statsbomb_xg', 0.0)
                outcome = shot.get('shot_outcome', {})
                outcome_name = outcome.get('name') if isinstance(outcome, dict) else str(outcome)
                is_goal = 1 if outcome_name == 'Goal' else 0

                shots_data.append({
                    'match_id': match_id,
                    'goalkeeper': gk_name,
                    'shot_x': shot_loc[0],
                    'shot_y': shot_loc[1],
                    'gk_x': gk_loc[0],
                    'gk_y': gk_loc[1],
                    'shot_dist': shot_dist,
                    'shot_angle': shot_angle,
                    'angle_zone': 'Central' if shot_angle >= 25.0 else 'Wide/Tight',
                    'bisector_dist': bisector_dist,
                    'gk_depth': gk_depth,
                    'occlusion_count': occlusion_count,
                    'pre_shot_xg': p_xg,
                    'is_goal': is_goal
                })
    except Exception:
        continue

df = pd.DataFrame(shots_data)

# Apply Spatial & Tracking Noise Filters (Abstract Specifications)
df = df[(df['shot_x'] >= 70.0) & (df['gk_depth'] <= 12.0)].reset_index(drop=True)
print(f"Dataset ready: {len(df)} shot observations across {df['match_id'].nunique()} matches.")

# ==============================================================================
# GROUPED CROSS-VALIDATION & PROBABILITY CALIBRATION PIPELINE
# ==============================================================================
features = ['shot_dist', 'bisector_dist', 'gk_depth', 'occlusion_count', 'pre_shot_xg']
X = df[features]
y = df['is_goal']
groups = df['match_id']

# Prepare arrays for Out-Of-Fold (OOF) store
df['model_goal_prob'] = np.nan
df['shap_bisector_impact'] = np.nan

gkf = GroupKFold(n_splits=5)

print("\n2. Executing 5-Fold Match-Grouped Cross-Validation with Isotonic Calibration...")
for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups=groups)):
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    # Base Classifier
    base_xgb = XGBClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric='logloss'
    )

    # Calibrated Classifier (Isotonic Regression on inner CV)
    calibrated_xgb = CalibratedClassifierCV(estimator=base_xgb, method='isotonic', cv=3)

    # Fit Base Model (for SHAP) and Calibrated Model (for Probabilities)
    base_xgb.fit(X_tr, y_tr)
    calibrated_xgb.fit(X_tr, y_tr)

    # Out-of-fold probability predictions
    oof_probs = calibrated_xgb.predict_proba(X_val)[:, 1]
    df.loc[val_idx, 'model_goal_prob'] = oof_probs

    # Calculate SHAP values out-of-sample using base XGBoost
    explainer = shap.TreeExplainer(base_xgb)
    shap_vals = explainer(X_val)
    df.loc[val_idx, 'shap_bisector_impact'] = shap_vals.values[:, features.index('bisector_dist')]

# Compute Metric: Positional Goals Saved Above Average (GSAA_pos)
df['positional_gsaa'] = df['pre_shot_xg'] - df['model_goal_prob']

# ==============================================================================
# PUBLICATION DIAGNOSTICS & METRICS REPORTING
# ==============================================================================
auc = roc_auc_score(df['is_goal'], df['model_goal_prob'])
loss = log_loss(df['is_goal'], df['model_goal_prob'])
brier = brier_score_loss(df['is_goal'], df['model_goal_prob'])

print("\n--- Model Evaluation (Out-Of-Fold Performance) ---")
print(f"ROC AUC:    {auc:.3f}")
print(f"Log Loss:   {loss:.3f}")
print(f"Brier Score: {brier:.3f}")

# Goalkeeper Aggregation Table for Paper
print("\n--- Sample Goalkeeper Leaderboard (Min. 30 Shots Facing) ---")
gk_summary = df.groupby('goalkeeper').agg(
    shots_faced=('is_goal', 'count'),
    actual_goals=('is_goal', 'sum'),
    mean_pre_xg=('pre_shot_xg', 'mean'),
    total_gsaa_pos=('positional_gsaa', 'sum'),
    mean_gsaa_pos=('positional_gsaa', 'mean')
).query('shots_faced >= 30').sort_values(by='total_gsaa_pos', ascending=False)

print(gk_summary.head(10))

# Export clean dataset for manuscript figures and Streamlit integration
df.to_csv("goalkeeper_shot_analysis_jsa.csv", index=False)
print("\nExported clean out-of-fold metrics to 'goalkeeper_shot_analysis_jsa.csv'.")

1. Pulling StatsBomb Open Data...
Dataset ready: 98871 shot observations across 3961 matches.

2. Executing 5-Fold Match-Grouped Cross-Validation with Isotonic Calibration...

--- Model Evaluation (Out-Of-Fold Performance) ---
ROC AUC:    0.807
Log Loss:   0.262
Brier Score: 0.075

--- Sample Goalkeeper Leaderboard (Min. 30 Shots Facing) ---
                          shots_faced  actual_goals  mean_pre_xg  \
goalkeeper                                                         
Ronwen Williams                    99             2     0.139720   
Zecira Musovic                    158             6     0.103025   
Yann Sommer                       287            32     0.109554   
Unai Simón Mendibil               187            19     0.129117   
Nelson de Jesus da Silva           32             4     0.109545   
Sergio Rochet Álvarez              82             5     0.099796   
Ingrid Hjelmseth                   81             6     0.111063   
Lionel Mpasi-Nzau                  49       